# TASK 2: Power BI Data Modeling & Preparation
### Star Schema Architecture, Referential Integrity Verification, and Dimensional Normalization

**Author**: Vijay Maheswari  
**Framework**: RISK Methodology — Solution Development & Engineering  
**Output Tables**: FactSales, DimProduct, DimCustomer, DimGeography, DimDate


In [1]:
import sys
import os

# Ensure parent directory is in sys.path
for p in [os.path.abspath(".."), os.path.abspath(".")]:
    if p not in sys.path:
        sys.path.insert(0, p)

import pandas as pd
import numpy as np
from src.loaders import load_superstore
from src.cleaning import clean_superstore
from src.build_powerbi_model import build_star_schema


## 1. Load Validated Retail Data
We load the validated retail sales records and inspect dimensional keys before decomposition.


In [2]:
raw_df = load_superstore()
clean_df = clean_superstore(raw_df)
print(f"Cleaned source records: {clean_df.shape}")


Cleaned source records: (9994, 31)


## 2. Dimensional Decomposition & Star Schema Generation
We construct `FactSales` (order line grain) and 4 conforming dimension tables (`DimProduct`, `DimCustomer`, `DimGeography`, `DimDate`).


In [3]:
tables = build_star_schema(clean_df)
for name, df_tbl in tables.items():
    print(f"{name:15}: {df_tbl.shape[0]:>6} rows | {df_tbl.shape[1]:>2} columns")


FactSales      :   9994 rows | 12 columns
DimProduct     :   1862 rows |  4 columns
DimCustomer    :    793 rows |  3 columns
DimGeography   :    631 rows |  5 columns
DimDate        :   1464 rows | 12 columns


## 3. Referential Integrity Audit
We verify that 100% of foreign keys in `FactSales` resolve to existing dimension keys (zero orphan records).


In [4]:
fact = tables["FactSales"]
dim_p = tables["DimProduct"]
dim_c = tables["DimCustomer"]
dim_g = tables["DimGeography"]
dim_d = tables["DimDate"]

print("Orphan Product IDs :", (~fact["Product ID"].isin(dim_p["Product ID"])).sum())
print("Orphan Customer IDs:", (~fact["Customer ID"].isin(dim_c["Customer ID"])).sum())
print("Orphan Postal Codes:", (~fact["Postal Code"].isin(dim_g["Postal Code"])).sum())
print("Orphan Order Dates :", (~fact["Order Date"].isin(dim_d["Date"])).sum())
print("Referential integrity test passed: 100% valid relationships!")


Orphan Product IDs : 0
Orphan Customer IDs: 0
Orphan Postal Codes: 0
Orphan Order Dates : 0
Referential integrity test passed: 100% valid relationships!


## 4. Reconcile Star Schema Aggregates Against Cleaned Source
We confirm that aggregating `FactSales` matches the single-table source down to the exact cent.


In [5]:
source_sales = clean_df["Sales"].sum()
source_profit = clean_df["Profit"].sum()
fact_sales = fact["Sales"].sum()
fact_profit = fact["Profit"].sum()

print(f"Source Sales: ${source_sales:,.2f} | Fact Sales: ${fact_sales:,.2f} | Match: {source_sales == fact_sales}")
print(f"Source Profit: ${source_profit:,.2f} | Fact Profit: ${fact_profit:,.2f} | Match: {source_profit == fact_profit}")


Source Sales: $2,297,200.86 | Fact Sales: $2,297,200.86 | Match: True
Source Profit: $286,397.02 | Fact Profit: $286,397.02 | Match: True
